# チュートリアル2: 条件付き生成

このチュートリアルでは、特定の性質を持つ分子を生成する方法を学びます。

**所要時間**: 20-30分

**学習内容**:
- 性質の正規化
- 条件付きモデルの学習
- ターゲット性質での生成
- 性質スイープ
- 複数性質での条件付け
- Classifier-Free Guidance

**前提知識**: チュートリアル1（基本的な分子生成）


## セットアップとインポート


In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from qm9 import dataset
from qm9.models import get_model
from qm9 import utils as qm9_utils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')


## 1. データ読み込みと性質の理解

QM9データセットには、各分子について複数の量子化学的性質が含まれています:

- **alpha**: 分極率（isotropic）
- **homo**: HOMOエネルギー
- **lumo**: LUMOエネルギー
- **gap**: HOMO-LUMOギャップ
- **mu**: 双極子モーメント
- **Cv**: 熱容量

これらの性質を条件として、望ましい特性を持つ分子を生成できます。


In [ ]:
# 条件付き生成の設定
class Args:
    def __init__(self):
        self.batch_size = 64
        self.num_workers = 0
        self.filter_n_atoms = None
        self.dataset = 'qm9'
        self.datadir = 'qm9/temp'
        self.conditioning = ['alpha']  # 分極率で条件付け
        self.remove_h = False

args = Args()

# データローダーの取得
print('データセットを読み込んでいます（条件付き性質付き）...')
dataloaders, charge_scale = dataset.retrieve_dataloaders(args)

print(f'条件付けに使用する性質: {args.conditioning}')
print(f'学習サンプル数: {len(dataloaders["train"].dataset)}')


### 性質分布の確認

条件付けに使用する性質の分布を確認します。


In [ ]:
# 性質の統計情報を収集
alpha_values = []

for batch in dataloaders['train']:
    if 'context' in batch and batch['context'] is not None:
        # context には正規化された性質値が含まれる
        alpha_values.extend(batch['context'][:, 0].cpu().numpy())
    if len(alpha_values) > 1000:  # サンプル数を制限
        break

alpha_values = np.array(alpha_values)

print(f'\n分極率 (alpha) の統計:')
print(f'  平均: {alpha_values.mean():.4f}')
print(f'  標準偏差: {alpha_values.std():.4f}')
print(f'  最小値: {alpha_values.min():.4f}')
print(f'  最大値: {alpha_values.max():.4f}')

# ヒストグラムの表示
plt.figure(figsize=(10, 4))
plt.hist(alpha_values, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('分極率 (alpha) - 正規化済み')
plt.ylabel('頻度')
plt.title('訓練データの分極率分布')
plt.grid(True, alpha=0.3)
plt.show()


## 2. 性質の正規化

性質値を平均0、標準偏差1に正規化します。これにより、学習の安定性が向上します。

正規化式: `normalized_value = (value - mean) / mad`

ここで、MAD (Median Absolute Deviation) を標準偏差の代わりに使用します。


In [ ]:
# 性質の平均とMADを計算
property_norms = qm9_utils.compute_mean_mad(dataloaders, args.conditioning, args.dataset)

print('性質の正規化パラメータ:')
for prop in args.conditioning:
    mean, mad = property_norms[prop]['mean'], property_norms[prop]['mad']
    print(f'  {prop}: 平均={mean:.4f}, MAD={mad:.4f}')


## 3. 条件付きモデルの構築と学習

条件付き情報を受け取るモデルを構築します。


In [ ]:
# モデル設定
args.n_epochs = 5  # チュートリアル用
args.lr = 1e-4
args.nf = 128
args.n_layers = 4
args.diffusion_steps = 500
args.diffusion_noise_schedule = 'polynomial_2'
args.diffusion_noise_precision = 1e-5
args.ema_decay = 0.999
args.normalize_factors = [1, 4, 1]
args.include_charges = True

# データセット情報
dataset_info = qm9_utils.get_dataset_info(args.dataset, args.remove_h)

# 条件付きモデルの構築
print('条件付きモデルを構築しています...')
model, nodes_dist, prop_dist = get_model(args, device, dataset_info, dataloaders['train'])

print(f'モデルパラメータ数: {sum(p.numel() for p in model.parameters()):,}')
print(f'条件次元: {len(args.conditioning)}')


In [ ]:
# 学習ループ
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-12)

print('条件付きモデルの学習を開始します...')
model.train()

for epoch in range(args.n_epochs):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch_idx, batch in enumerate(dataloaders['train']):
        # データをデバイスに転送
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                for k, v in batch.items()}
        
        # 順伝播（contextが自動的に使用される）
        optimizer.zero_grad()
        loss = model(batch)
        
        # 逆伝播
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        if (batch_idx + 1) % 100 == 0:
            print(f'  エポック {epoch+1}/{args.n_epochs}, '
                  f'バッチ {batch_idx+1}, '
                  f'損失: {loss.item():.4f}')
        
        if batch_idx >= 200:
            break
    
    avg_loss = epoch_loss / n_batches
    print(f'エポック {epoch+1}/{args.n_epochs} 完了, 平均損失: {avg_loss:.4f}')

model.eval()
print('学習完了!')


## 4. ターゲット性質での分子生成

特定の分極率を持つ分子を生成します。


In [ ]:
# ターゲット分極率の設定（正規化済み）
target_alpha_norm = 0.5  # 正規化された値

# 実際の値に逆変換
mean = property_norms['alpha']['mean']
mad = property_norms['alpha']['mad']
target_alpha_real = target_alpha_norm * mad + mean

print(f'ターゲット分極率:')
print(f'  正規化値: {target_alpha_norm}')
print(f'  実際の値: {target_alpha_real:.4f}')

# サンプリング
n_samples = 10

with torch.no_grad():
    # ノード数分布からサンプリング
    n_nodes = nodes_dist.sample(n_samples)
    max_n_nodes = int(n_nodes.max().item())
    
    # マスクの作成
    arange = torch.arange(max_n_nodes, device=device).unsqueeze(0).expand(n_samples, -1)
    node_mask = arange < n_nodes.unsqueeze(1)
    edge_mask = node_mask.unsqueeze(1) * node_mask.unsqueeze(2)
    diag_mask = ~torch.eye(max_n_nodes, device=device, dtype=torch.bool).unsqueeze(0)
    edge_mask = edge_mask * diag_mask
    
    # 条件の作成（全サンプルで同じターゲット値）
    context = torch.ones(n_samples, 1, device=device) * target_alpha_norm
    
    # 条件付きサンプリング
    print(f'\nターゲット分極率={target_alpha_real:.4f}で分子を生成中...')
    x, h = model.sample(n_samples, max_n_nodes, node_mask, edge_mask, context=context)
    
    print('条件付き生成完了!')


## 5. 性質スイープ

異なる分極率値で複数の分子を生成し、性質の変化を観察します。


In [ ]:
# 性質値の範囲を設定
alpha_values_norm = np.linspace(-1.0, 1.0, 5)  # 正規化された値

print('性質スイープを実行中...')
print(f'分極率値: {alpha_values_norm}')

results = []

with torch.no_grad():
    for alpha_norm in alpha_values_norm:
        # 各性質値で分子を生成
        n_samples_sweep = 5
        n_nodes = nodes_dist.sample(n_samples_sweep)
        max_n_nodes = int(n_nodes.max().item())
        
        arange = torch.arange(max_n_nodes, device=device).unsqueeze(0).expand(n_samples_sweep, -1)
        node_mask = arange < n_nodes.unsqueeze(1)
        edge_mask = node_mask.unsqueeze(1) * node_mask.unsqueeze(2)
        diag_mask = ~torch.eye(max_n_nodes, device=device, dtype=torch.bool).unsqueeze(0)
        edge_mask = edge_mask * diag_mask
        
        context = torch.ones(n_samples_sweep, 1, device=device) * alpha_norm
        
        x, h = model.sample(n_samples_sweep, max_n_nodes, node_mask, edge_mask, context=context)
        
        # 結果を保存
        alpha_real = alpha_norm * mad + mean
        avg_atoms = node_mask.sum(dim=1).float().mean().item()
        
        results.append({
            'alpha_norm': alpha_norm,
            'alpha_real': alpha_real,
            'avg_atoms': avg_atoms,
            'positions': x,
            'features': h
        })
        
        print(f'  alpha={alpha_real:.4f}: 平均原子数={avg_atoms:.1f}')

print('\n性質スイープ完了!')


## 6. 複数性質での条件付け

複数の性質を同時に指定して分子を生成します。


In [ ]:
# 複数性質での条件付け設定
print('複数性質での条件付けの例:')
print('\n注意: このデモでは1つの性質のみを使用していますが、')
print('args.conditioning = ["alpha", "gap", "homo"]のように複数指定可能です。')
print('\nその場合、contextは [n_samples, n_properties] の形状になります。')
print('\n例:')
print('  context = torch.tensor([[0.5, -0.3, 0.2], ...])  # [alpha, gap, homo]')
print('\nこれにより、分極率、HOMO-LUMOギャップ、HOMOエネルギーを')
print('同時に制御した分子生成が可能になります。')


## 7. Classifier-Free Guidance

Classifier-Free Guidance は、条件付き生成の強度を調整する手法です。

**数式**: `output = unconditional + guidance_scale * (conditional - unconditional)`

- `guidance_scale = 0`: 無条件生成
- `guidance_scale = 1`: 通常の条件付き生成
- `guidance_scale > 1`: より強い条件付け


In [ ]:
print('Classifier-Free Guidance の概念:')
print('\n実装には以下が必要です:')
print('1. 学習時: ランダムに条件をドロップ（例: 10%の確率）')
print('2. サンプリング時: 無条件と条件付きの両方を計算')
print('3. 結果を guidance_scale でブレンド')
print('\nこの手法により、生成される分子がターゲット性質に')
print('より忠実になります。')
print('\n注意: 完全な実装には追加のコード修正が必要です。')


## 8. 性質予測による検証

生成された分子が、実際にターゲット性質を持っているかを検証します。

**注意**: 実際の性質値を正確に予測するには、別途性質予測器を学習する必要があります。


In [ ]:
print('性質予測による検証:')
print('\n生成された分子の性質を検証するには:')
print('1. 性質予測器を学習（train_property_predictor.py）')
print('2. 生成された分子の性質を予測')
print('3. ターゲット値と比較')
print('\nコマンド例:')
print('  python train_property_predictor.py --property alpha')
print('  python eval_conditional_qm9.py --model_path outputs/model --property alpha')
print('\nこれにより、条件付き生成の精度を定量的に評価できます。')


## まとめ

このチュートリアルでは、以下を学習しました:

✅ 性質の正規化とその重要性  
✅ 条件付きモデルの構築と学習  
✅ ターゲット性質での分子生成  
✅ 性質スイープによる探索  
✅ 複数性質での条件付けの概念  
✅ Classifier-Free Guidance の原理  
✅ 生成結果の検証方法  

### 次のステップ

- **チュートリアル3**: 結晶生成 - 周期境界条件と空間群
- **チュートリアル4**: 分子記述子とASE - カスタム条件付け
- **実践**: 複数性質での条件付け生成を実装

### 重要な原則

1. **厳密な正規化**: 性質値を適切にスケーリング
2. **明示的な条件**: フォールバックなしの明確な条件付け
3. **検証可能性**: 生成結果を定量的に評価

---

**質問やフィードバックは、GitHubのIssuesでお寄せください。**
